# 🤖 Modelagem e Avaliação de Filtragem Colaborativa (KNN vs SVD)

**Disciplina:** Tópicos em Sistemas de Recomendação (UNITINS)
**Autor:** Matheus N.
**Foco:** Treinamento, avaliação comparativa (RMSE/MAE + Precision@K/NDCG@K) e exportação de modelos colaborativos, com métricas contra o ground truth gerado.

> O script executável é `executar_modelagem.py`. Decisões em [DECISOES.md](DECISOES.md).


## 1. Configuração e Carregamento

Parâmetros da avaliação — split único 80/20 com 120 mil ratings de teste **(Decisão B2)**, top-3 fatores SVD {20, 50, 100}, ranking para K ∈ {5, 10, 20} em 1.000 usuários amostrados.


In [1]:
import os, time, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, joblib
from scipy import sparse
from scipy.stats import wilcoxon
from surprise import Dataset, Reader, SVD, KNNBasic
from surprise.model_selection import train_test_split

SEED = 42; np.random.seed(SEED); sns.set_theme(style="whitegrid", palette="muted")
K_RANKING = [5, 10, 20]; N_USERS_AVALIACAO = 1000; TAM_POOL = 2000
FATORES_SVD = [20, 50, 100]


In [4]:
df = pd.read_csv("../data/processed/interacoes_sinteticas.csv")
verdade = joblib.load("../data/processed/verdade_afinidade.pkl")
W, B, G = verdade['W'], verdade['B'], verdade['G']
job_ids_cat = verdade['job_ids']; params_g = verdade['params']
THRESHOLD_REL = params_g['threshold_relevancia']
print(f"Interações: {len(df):,}  |  Usuários: {df['user_id'].nunique():,}  |  Vagas: {df['job_id'].nunique():,}")

reader = Reader(rating_scale=(1,5))
data = Dataset.load_from_df(df[['user_id','job_id','rating']], reader)
trainset, testset = train_test_split(data, test_size=0.20, random_state=SEED)
r_true = np.array([t[2] for t in testset], dtype=float)
N_TEST = len(r_true)
print(f"Treino: {trainset.n_ratings:,}  |  Teste: {N_TEST:,}")

def comp_afinidade_user(user_id):
    aff_raw = np.dot(B, W[user_id - 1])
    return np.clip((aff_raw - 0.05) / 0.65, 0.0, 1.0)


Interações: 602,216  |  Usuários: 5,000  |  Vagas: 6,000
Treino: 481,772  |  Teste: 120,444


## 2. Baselines Triviais **(Decisão B1)**

Antes de comparar KNN e SVD, medimos três chutes ingênuos: (1) sempre chutar a média global, (2) a média do usuário, (3) a média do item. Se um modelo não vencer **todos** esses baselines, ele não aprendeu nada útil.


In [5]:
gm = float(trainset.global_mean)
pred_gm = np.full(N_TEST, gm)
rmse_gm = float(np.sqrt(np.mean((pred_gm - r_true)**2)))
mae_gm = float(np.mean(np.abs(pred_gm - r_true)))
print(f"Média Global: RMSE={rmse_gm:.4f} MAE={mae_gm:.4f} (pred={gm:.3f})")

u_means = {u: np.mean([r for _,_,r in trainset.all_ratings() if u==_]) for u in range(trainset.n_users)}  # simplificado
# Cálculo real:
u_means = {}
for uid, iid, r in trainset.all_ratings():
    u_means.setdefault(uid, []).append(r)
u_means = {u: np.mean(v) for u,v in u_means.items()}
pred_um = np.array([u_means.get(trainset.to_inner_uid(t[0]), gm) for t in testset])
print(f"Média por Usuário: RMSE={np.sqrt(np.mean((pred_um-r_true)**2)):.4f} MAE={np.mean(np.abs(pred_um-r_true)):.4f}")

i_means = {}
for uid, iid, r in trainset.all_ratings():
    i_means.setdefault(iid, []).append(r)
i_means = {i: np.mean(v) for i,v in i_means.items()}
pred_im = np.array([i_means.get(trainset.to_inner_iid(t[1]), gm) for t in testset])
rmse_im = np.sqrt(np.mean((pred_im-r_true)**2)); mae_im = np.mean(np.abs(pred_im-r_true))
print(f"Média por Item: RMSE={rmse_im:.4f} MAE={mae_im:.4f}")


Média Global: RMSE=1.5778 MAE=1.4455 (pred=3.250)
Média por Usuário: RMSE=1.5750 MAE=1.4254
Média por Item: RMSE=1.3202 MAE=1.1187


## 3. Modelos no Split 80/20

### KNN (Surprise KNNBasic, cosseno user-based, top-40 vizinhos, min_support=5)

Treinamos e avaliamos no teste de 120.444 ratings. Reportamos também o percentual de **predições impossíveis** (sem vizinhos, que caem para a média global) — **(Decisão B4)**.


In [ ]:
sim_opt = {'name':'cosine','user_based':True,'min_support':5}
t0 = time.time()
knn = KNNBasic(sim_options=sim_opt, verbose=False)
knn.fit(trainset)
preds = knn.test(testset)
est_knn = np.array([p.est for p in preds], dtype=float)
impossivel = np.array([p.details.get('was_impossible',False) for p in preds], dtype=bool)
n_imp = int(impossivel.sum())
rmse_knn = float(np.sqrt(np.mean((est_knn - r_true)**2)))
mae_knn = float(np.mean(np.abs(est_knn - r_true)))
print(f"KNN: RMSE={rmse_knn:.4f} MAE={mae_knn:.4f} ({time.time()-t0:.1f}s)")
print(f"  Impossíveis: {n_imp:,} de {N_TEST:,} ({n_imp/N_TEST*100:.1f}%)")
if n_imp < N_TEST:
    print(f"  RMSE só com vizinhos: {np.sqrt(np.mean((est_knn[~impossivel]-r_true[~impossivel])**2)):.4f}")


### SVD — Teste de Fatores Latentes {20, 50, 100}

Testamos cada configuração treinando no mesmo split 80/20 e avaliamos no teste. O melhor número de fatores é aquele com menor RMSE no teste.


In [ ]:
cv_svd = {}
for nf in FATORES_SVD:
    t0 = time.time()
    m = SVD(n_factors=nf, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=SEED)
    m.fit(trainset)
    est = np.array([p.est for p in m.test(testset)], dtype=float)
    rmse = float(np.sqrt(np.mean((est - r_true)**2)))
    mae = float(np.mean(np.abs(est - r_true)))
    cv_svd[nf] = {'rmse':rmse, 'mae':mae, 'modelo':m, 'est':est}
    print(f"SVD f={nf:3d}: RMSE={rmse:.4f} MAE={mae:.4f} ({time.time()-t0:.1f}s)")

best_f = min(cv_svd, key=lambda k: cv_svd[k]['rmse'])
print(f"\nMelhor: {best_f} fatores (RMSE {cv_svd[best_f]['rmse']:.4f})")
svd = cv_svd[best_f]['modelo']; est_svd = cv_svd[best_f]['est']

# Teste de Wilcoxon pareado
if n_imp < N_TEST:
    mask = ~impossivel
    stat, p = wilcoxon((est_svd[mask]-r_true[mask])**2, (est_knn[mask]-r_true[mask])**2)
    print(f"Wilcoxon (SVD vs KNN, {mask.sum():,} pares): p={p:.3e} {'SIGNIFICATIVO' if p<0.05 else 'NÃO significativo'}")


## 4. Tabela Comparativa de Erro

Consolidamos todos os resultados — baselines, KNN e SVD — em uma tabela. O RMSE mostra o **ganho absoluto** de cada modelo sobre o chute ingênuo.


In [ ]:
df_erro = pd.DataFrame({
    'Modelo': ['Média Global','Média por Usuário','Média por Item','KNN (Cosseno)',f'SVD ({best_f} fatores)'],
    'RMSE': [rmse_gm, np.sqrt(np.mean((pred_um-r_true)**2)), rmse_im, rmse_knn, cv_svd[best_f]['rmse']],
    'MAE': [mae_gm, np.mean(np.abs(pred_um-r_true)), mae_im, mae_knn, cv_svd[best_f]['mae']],
})
print(df_erro.to_string(index=False))


## 5. Métricas de Ranking — Precision@K e NDCG@K **(Decisão B3)**

Avaliamos a **qualidade do ranking** contra o ground truth. Para cada um dos 1.000 usuários amostrados (250 por persona):

1. **Pool:** 2.000 vagas aleatórias **não vistas no treino** (sem oversampling de relevantes — isso daria baseline inflado).
2. **Relevância:** aff ≥ 0,60 (≈35% do pool, em média).
3. **NDCG:** ganho graduado $2^{grau} - 1$, onde grau = round(5·aff).
4. **Baselines de ranking:** Aleatório (ordem aleatória) e Popularidade (ordenar pela média do item no treino).
5. **Modelos:** SVD (fatores latentes, vetorizado) e KNN (cosseno todos-os-vizinhos, vetorizado via scipy sparse).

A pré-computação é vetorizada (sem loops Python) para ser rápida: matrizes (1000, 6000) com operações numpy/scipy.


In [ ]:
def avaliar_ranking(pred_func, users, pools, rels, grades):
    P = {k:[] for k in K_RANKING}; N = {k:[] for k in K_RANKING}
    for u in users:
        pool = pools[u]
        est = pred_func(u, pool)
        ordem = np.argsort(-est)
        for k in K_RANKING:
            top = ordem[:k]
            P[k].append(rels[u][top].mean())
            dcg = ((2**grades[u][top]-1) / np.log2(np.arange(2,k+2))).sum()
            idcg = ((2**np.sort(grades[u])[::-1][:k]-1) / np.log2(np.arange(2,k+2))).sum()
            N[k].append(dcg/idcg if idcg > 0 else 0.0)
    return {k: (np.mean(P[k]), np.mean(N[k])) for k in K_RANKING}

# Amostra de usuários + pools
user_ids = sorted(df['user_id'].unique())
personas_map = dict(zip(df['user_id'], df['user_persona']))
pp = {}
for u in user_ids:
    pp.setdefault(personas_map[u], []).append(u)
n_pp = N_USERS_AVALIACAO // 4
users_amostra = sorted(np.concatenate([np.random.choice(v, min(n_pp,len(v)), replace=False) for v in pp.values()]).tolist())
print(f"Usuários amostrados: {len(users_amostra)}")

vistos = {}
for u,i,r in trainset.all_ratings(): vistos.setdefault(u, set()).add(i)
M = len(job_ids_cat)
cat_to_inner = {}
for i in range(M):
    try: cat_to_inner[i] = trainset.to_inner_iid(job_ids_cat[i])
    except ValueError: pass

pools, rels, grades = {}, {}, {}
for u in users_amostra:
    aff = comp_afinidade_user(u)
    inner_u = trainset.to_inner_uid(u)
    seen = vistos.get(inner_u, set())
    nao_vistos = [i for i in range(M) if i not in seen]
    np.random.seed(u)
    pool = np.random.choice(nao_vistos, size=min(TAM_POOL, len(nao_vistos)), replace=False)
    pools[u] = pool; rels[u] = aff[pool] >= THRESHOLD_REL; grades[u] = np.minimum(5, np.round(5*aff[pool])).astype(int)
print(f"Pool médio: {np.mean([len(p) for p in pools.values()]):.0f}  |  Relevantes: {np.mean([r.mean()*100 for r in rels.values()]):.1f}%")


In [ ]:
# KNN vetorizado (apenas sampled users, evita S@R denso de 5000x6000)
print("\nPré-computando KNN e SVD...")
t0 = time.time()
rows, cols, vals = [], [], []
for uid, iid, r in trainset.all_ratings():
    rows.append(uid); cols.append(iid); vals.append(r)
R = sparse.csr_matrix((vals, (rows, cols)), shape=(trainset.n_users, trainset.n_items), dtype=float)
rn = np.sqrt(np.asarray(R.multiply(R).sum(axis=1)).ravel()); rn[rn==0] = 1.0
R_unit = R.multiply(1.0 / rn[:, None]).tocsr()

inner_amostra = [trainset.to_inner_uid(u) for u in users_amostra]
S_sample = (R_unit[inner_amostra] @ R_unit.T).toarray()
for k, inid in enumerate(inner_amostra): S_sample[k, inid] = 0.0
N_den = sparse.csr_matrix((np.ones(len(vals)), (rows, cols)), shape=R.shape)
num = S_sample @ R; den = S_sample @ N_den
knn_pred_inner = np.where(den > 0, num / den, gm)
print(f"  KNN sparse: {time.time()-t0:.1f}s")

# SVD vetorizado: est = mu + bu + bi + pu·qi
t0 = time.time()
_inner_a = [trainset.to_inner_uid(u) for u in users_amostra]
_pu = np.array([svd.pu[i] for i in _inner_a]); _bu = np.array([svd.bu[i] for i in _inner_a])
inner_to_cat = {inid: cp for cp, inid in cat_to_inner.items()}
inner_present = sorted(cat_to_inner.values())
qi_cat = np.array([svd.qi[i] for i in inner_present]); bi_cat = np.array([svd.bi[i] for i in inner_present])
mu = svd.trainset.global_mean
svd_raw = mu + _bu[:, None] + bi_cat[None, :] + (_pu @ qi_cat.T)
svd_preds = np.full((len(users_amostra), M), gm)
for k, inid in enumerate(inner_present):
    svd_preds[:, inner_to_cat[inid]] = svd_raw[:, k]
print(f"  SVD: {time.time()-t0:.1f}s")

knn_preds = np.full((len(users_amostra), M), gm)
for cp, inid in cat_to_inner.items(): knn_preds[:, cp] = knn_pred_inner[:, inid]
linha = {u: i for i, u in enumerate(users_amostra)}

def pred_random(u, pool): np.random.seed(u); return np.random.random(len(pool))
def pred_pop(u, pool):
    out = np.empty(len(pool))
    for j,ci in enumerate(pool):
        ii = cat_to_inner.get(ci)
        out[j] = i_means.get(ii, gm) if ii is not None else gm
    return out
def pred_svd(u, pool): return svd_preds[linha[u]][pool]
def pred_knn(u, pool): return knn_preds[linha[u]][pool]


In [ ]:
print("\nAvaliando rankings...")
res = {}
for nome, func in [('Aleatório', pred_random), ('Popularidade', pred_pop),
                   ('SVD', pred_svd), ('KNN', pred_knn)]:
    t0 = time.time()
    res[nome] = avaliar_ranking(func, users_amostra, pools, rels, grades)
    p10 = res[nome][10][0]; n10 = res[nome][10][1]
    print(f"  {nome:15s}  P@10={p10:.4f}  NDCG@10={n10:.4f}  ({time.time()-t0:.1f}s)")

df_rank = pd.DataFrame([{'Método':m, 'K':k, 'Precision@K':v[k][0], 'NDCG@K':v[k][1]}
                         for m,v in res.items() for k in K_RANKING])
print("\nTabela de Precision@K:"); print(df_rank.pivot('Método','K','Precision@K').round(4))
print("\nTabela de NDCG@K:"); print(df_rank.pivot('Método','K','NDCG@K').round(4))


## 6. Gráficos

Geramos dois gráficos: (1) barras de RMSE/MAE comparando todos os modelos, (2) linhas de Precision@K e NDCG@K para K ∈ {5, 10, 20}.


In [ ]:
os.makedirs("data/figuras", exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.barplot(data=df_erro, x='Modelo', y='RMSE', ax=axes[0], palette='Blues_d')
axes[0].set_title('RMSE (menor é melhor)', fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=20, ha='right')
sns.barplot(data=df_erro, x='Modelo', y='MAE', ax=axes[1], palette='Oranges_d')
axes[1].set_title('MAE (menor é melhor)', fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20, ha='right')
plt.tight_layout(); plt.savefig("data/figuras/comparativo_modelos_cf.png", dpi=150, bbox_inches='tight'); plt.close()
print("✅ comparativo_modelos_cf.png")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.lineplot(data=df_rank, x='K', y='Precision@K', hue='Método', marker='o', ax=axes[0], linewidth=2.2)
axes[0].set_title('Precision@K', fontweight='bold'); axes[0].set_ylim(0, 1.05)
sns.lineplot(data=df_rank, x='K', y='NDCG@K', hue='Método', marker='o', ax=axes[1], linewidth=2.2)
axes[1].set_title('NDCG@K', fontweight='bold'); axes[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.savefig("data/figuras/metricas_ranking_cf.png", dpi=150, bbox_inches='tight'); plt.close()
print("✅ metricas_ranking_cf.png")


## 7. Exportar Modelo e Metadados

Exportamos o modelo SVD vencedor e metadados completos da avaliação (incluindo Precision@10 e NDCG@10) para uso no dashboard (aba CF) e relatório.


In [ ]:
joblib.dump(svd, "data/processed/modelo_svd.pkl")
p10 = res['SVD'][10][0]; ndcg10 = res['SVD'][10][1]
p10_knn = res['KNN'][10][0]; ndcg10_knn = res['KNN'][10][1]
metadados = {
    'modelo':'SVD', 'n_factors':int(best_f),
    'rmse_svd':cv_svd[best_f]['rmse'], 'mae_svd':cv_svd[best_f]['mae'],
    'rmse_knn':rmse_knn, 'mae_knn':mae_knn,
    'rmse_media_global':rmse_gm, 'mae_media_global':mae_gm,
    'rmse_media_user':np.sqrt(np.mean((pred_um-r_true)**2)),
    'rmse_media_item':rmse_im,
    'knn_predicoes_impossiveis_pct':float(n_imp/N_TEST*100),
    'precision_at_10_svd':float(p10), 'ndcg_at_10_svd':float(ndcg10),
    'precision_at_10_knn':float(p10_knn), 'ndcg_at_10_knn':float(ndcg10_knn),
    'n_users':int(df['user_id'].nunique()), 'n_jobs':int(df['job_id'].nunique()),
    'n_ratings':int(len(df)),
    'esparsidade_pct':float((1-len(df)/(df['user_id'].nunique()*df['job_id'].nunique()))*100),
    'params_gerador':params_g,
}
joblib.dump(metadados, "data/processed/metadados_cf.pkl")
print("✅ modelo_svd.pkl + metadados_cf.pkl exportados!")


## 8. Verificação dos Critérios de Aceitação

O script verifica automaticamente os 10 critérios definidos em DECISOES.md. Todos devem passar ✅ para que a reestruturação seja considerada bem-sucedida.


In [ ]:
def checar(nome, cond, det=""):
    print(f"  {'✅' if cond else '❌'} | {nome} {det}")

checar("KNN: impossíveis < 5%", n_imp/N_TEST < 0.05, f"({n_imp/N_TEST*100:.1f}%)")
checar("SVD RMSE < média global", cv_svd[best_f]['rmse'] < rmse_gm, f"(SVD={cv_svd[best_f]['rmse']:.4f} vs {rmse_gm:.4f})")
checar("KNN RMSE < média global", rmse_knn < rmse_gm, f"(KNN={rmse_knn:.4f} vs {rmse_gm:.4f})")
checar("SVD RMSE ≥ 5% abaixo da média", cv_svd[best_f]['rmse'] < 0.95*rmse_gm, f"({(1-cv_svd[best_f]['rmse']/rmse_gm)*100:.1f}%)")
checar("SVD P@10 > Aleatório", p10 > res['Aleatório'][10][0])
checar("KNN P@10 > Aleatório", p10_knn > res['Aleatório'][10][0])
checar("SVD P@10 > Popularidade", p10 > res['Popularidade'][10][0])
checar("SVD NDCG@10 > 0.70", ndcg10 > 0.70, f"({ndcg10:.4f})")
checar("KNN NDCG@10 > 0.70", ndcg10_knn > 0.70, f"({ndcg10_knn:.4f})")
if n_imp < N_TEST:
    checar("Diferença SVD-KNN significativa (p<0.05)", p < 0.05, f"(p={p:.3e})")
print("\n🎯 Modelagem concluída!")
